<a href="https://colab.research.google.com/github/eshikanahata/DC-Mini-Project/blob/Nikhil/AI_DC_Task3_Nikhil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The RAG Pipeline (Chunking -> Vector Store -> Generation)

Objective:  
You will learn why "how you read" data (Chunking) matters as much as "what you read," and you will build a Retrieval Augmented Generation (RAG) pipeline.  
  



# Section 0: Setup & Prerequisites

Install the necessary libraries. You will likely need langchain, langchain-community, chromadb, and an embedding provider.

In [ ]:
# INITIAL SETUP (RUN THIS ONCE)

# 1. Install dependencies with specific versions to avoid conflicts
!pip install -q -U \
  torch \
  transformers \
  sentence-transformers \
  accelerate \
  bitsandbytes \
  langchain \
  langchain-community \
  chromadb \
  pysqlite3-binary

# 2. Fix Colab's SQLite version issue (Must happen before importing chromadb)
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# 3. Check if GPU is available
import torch
if not torch.cuda.is_available():
    print("WARNING: You are running on CPU. Go to Runtime -> Change runtime type -> T4 GPU")
else:
    print(f" GPU Detected: {torch.cuda.get_device_name(0)}")

In [ ]:
# TODO: Import any other necessary libraries here
# !pip install ...
!pip install langchain
!pip install langchain-text-splitters

# Section 1: Chunking Experiment

LLMs have context windows. We must slice our data. But if you slice a sentence in half, the meaning might be lost. Let's prove this.

In [ ]:
# Load a text file of your choice
from google.colab import files
uploaded = files.upload()

def load_data(path):
    with open(path, "r") as f:
        return f.read()

raw_text = load_data("TechNova Inc. - Company Overview.txt")
print(f"Loaded {len(raw_text)} characters.")

Implement a splitter that strictly cuts text every x characters, regardless of sentence boundaries.

In [ ]:
def naive_splitter(text, chunk_size=500):
    """
    Splits text strictly by character count.
    Returns: List[str]
    """
    # TODO: Implement strictly fixed-size splitting WITHOUT using a library (use pure Python)
    chunks = []
    for i in range(0,len(text),chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

    pass

naive_chunks = naive_splitter(raw_text)

Use a library (like LangChain) to split by "separators" (Paragraphs $\rightarrow$ Sentences $\rightarrow$ Words) to preserve meaning.

In [ ]:
!pip install langchain langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

specific_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

semantic_chunks = specific_splitter.split_text(raw_text)
print(f"Created {len(semantic_chunks)} chunks.")

In [ ]:
# TODO: Initialize a RecursiveCharacterTextSplitter
# Docs: Look up LangChain Text Splitters
# Constraints: Chunk size x, Chunk Overlap t


#semantic_chunks = [] # TODO: specific_splitter.split_text(raw_text)
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize a RecursiveCharacterTextSplitter
specific_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

semantic_chunks = specific_splitter.split_text(raw_text)

Find a specific example where the Naive splitter broke a sentence in half, rendering it meaningless, but the Semantic splitter kept it intact

In [ ]:
def find_broken_context(naive_list, semantic_list):
    """
    Print a side-by-side comparison of a specific segment where
    Naive failed and Semantic succeeded.
    """
    def find_broken_context(naive_list, semantic_list):
      """
      Print a side-by-side comparison of a specific segment where
      Naive failed and Semantic succeeded.
      """
      for i, chunk in enumerate(naive_list):
        if not chunk.rstrip().endswith(('.', '?', '!', ':', '"', "'")):
            print("NAIVE CHUNK (broken context):")
            print(chunk)
            print("\nNEAREST SEMANTIC CHUNK (for comparison):")
            # Find the semantic chunk that contains the start of this naive chunk
            target = chunk[:50]  # use the first 50 chars as a search key
            match = next((s for s in semantic_list if target[:20] in s), semantic_list[i])
            print(match)
            break

find_broken_context(naive_chunks, semantic_chunks)

In a text cell below, explain why the overlap parameter in the recursive splitter is essential for retrieval tasks.

# Section 2: Vector Storage

We will convert our semantic_chunks into vector embeddings and store them.

Initialize Embeddings & DB:
- You may use OpenAI Embeddings (if you have a key) or HuggingFace (all-MiniLM-L6-v2) for a free, local alternative.
- Use ChromaDB or FAISS as your store.

In [ ]:
!pip install langchain-huggingface langchain-chroma

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize your Embedding Model
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a Vector Store from your semantic_chunks
vector_db = Chroma.from_texts(
    texts=semantic_chunks,
    embedding=embedding_function
)

print("Vector Store successfully created.")

# Section 3: The MVP (Retrieval Loop)

We have the brain (LLM) and the memory (Vector DB). Now we need to wire them together.

Create a function that takes a user query, converts it to a vector, and finds the top 3 most relevant chunks from your database.

In [ ]:
def retrieve_context(query, k=3):
    """
    Args:
        query (str): The user's question
        k (int): Number of chunks to retrieve
    Returns:
        List[str]: The top k context chunks
    """
    # TODO: Use your vector_db to perform a similarity search
    results = vector_db.similarity_search(query, k=k)
    return [doc.page_content for doc in results]

# Test it
test_query = ""
context_results = retrieve_context(test_query)
print(f"Retrieved {len(context_results)} chunks.")

In [ ]:
!pip install transformers accelerate

Construct the final prompt. You must inject the retrieved context into the system prompt so the LLM answers only based on that data.

In [ ]:
llm = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def generate_answer(query):
    context_chunks = retrieve_context(query)
    context_str = "\n".join(context_chunks)

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context: {context_str}

Question: {query}

Answer:"""

    response = llm(prompt, max_new_tokens=200)[0]["generated_text"]
    return response

answer = generate_answer("Who founded TechNova?")
print(answer)

# The End (of task 3)